In [5]:
!pip install -q --force-reinstall "numpy==1.26.4"
!pip install -q --force-reinstall --no-deps rdkit
!pip install -q "scikit-learn==1.2.2" PyTDC chembl_webresource_client pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
esda 2.9.0 requires scikit-learn>=1.4, but you have scikit-learn 1.2.2 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
libpysal 4.14.1 requires scik

In [6]:
import numpy as np
print("numpy:", np.__version__)
import rdkit
print("RDKit OK:", rdkit.__version__)
import pandas as pd
print("pandas OK:", pd.__version__)
from chembl_webresource_client.new_client import new_client
print("ChEMBL client OK")

numpy: 1.26.4
RDKit OK: 2023.09.6
pandas OK: 2.2.2
ChEMBL client OK


In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/herg_hackathon'
os.makedirs(SAVE_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
from chembl_webresource_client.new_client import new_client
import pandas as pd
from datetime import date

HERG_TARGET_ID = "CHEMBL240"
activity = new_client.activity

print(f"Querying ChEMBL for target {HERG_TARGET_ID} ...")
results = activity.filter(
    target_chembl_id=HERG_TARGET_ID,
    standard_type__in=["IC50", "Ki", "EC50"],
    standard_units="nM",
    standard_value__isnull=False,
).only(
    "molecule_chembl_id", "canonical_smiles", "standard_type",
    "standard_value", "standard_units", "standard_relation",
    "assay_description", "assay_chembl_id", "document_chembl_id",
)

records = list(results)
print(f"Fetched {len(records)} raw bioactivity records from ChEMBL.")

chembl_df = pd.DataFrame(records)
chembl_df = chembl_df.rename(columns={"molecule_chembl_id": "chembl_id", "canonical_smiles": "smiles"})
chembl_df["source"] = "ChEMBL"
chembl_df["source_version"] = "ChEMBL_API_live"
chembl_df["pull_date"] = str(date.today())
chembl_df = chembl_df.dropna(subset=["smiles", "standard_value"])
chembl_df.insert(0, "compound_id", ["CMPD_" + str(i).zfill(5) for i in range(1, len(chembl_df) + 1)])

chembl_path = f"{SAVE_DIR}/raw_chembl_herg.csv"
chembl_df.to_csv(chembl_path, index=False)
print(f"Saved {len(chembl_df)} records to {chembl_path}")
print(f"Unique compounds: {chembl_df['chembl_id'].nunique()}")

Querying ChEMBL for target CHEMBL240 ...
Fetched 20075 raw bioactivity records from ChEMBL.
Saved 20073 records to /content/drive/MyDrive/herg_hackathon/raw_chembl_herg.csv
Unique compounds: 16215
